In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np

# Step 2: Sample Historical Data Setup (To train/fit the preprocessing state)
# This represents the historical data format used to establish column schemas
historical_csv = """Age,Gender,College_Tier,Specialization,CGPA,DSA_Problems_Solved,Internships,Certifications,Projects_Count,Communication_Skills,Aptitude_Test_Score,LeetCode_Rating,GitHub_Contributions,Hackathons_Participated,AI_ML_Skill_Level,System_Design_Knowledge,Resume_Score,Mock_Interview_Score,Placement_Status,Salary_LPA
19,Male,Tier 3,AI/ML,7.7,229,0,1,11,87,46,1676,241,6,1,1,50,62,1,9.99
21,Other,Tier 3,Civil,6.97,486,0,10,12,74,88,1718,82,3,8,10,62,35,1,13.06
20,Other,Tier 2,Electronics,6.74,180,0,5,2,45,83,1534,0,1,6,6,83,68,1,7.74
18,Female,Tier 1,Computer Science,6.64,98,4,6,2,75,72,1559,47,5,10,4,90,43,1,12.29"""

import io
df_train = pd.read_csv(io.StringIO(historical_csv))


# Step 3: Define Integrated Preprocessing Pipeline Object
class PipelineManager:
    def __init__(self, train_df):
        # Drop target columns from training frame schema to match user feature inputs
        features_only = train_df.drop(columns=['Placement_Status', 'Salary_LPA'], errors='ignore')
        
        # Calculate imputation baselines from historical data
        self.numerical_defaults = features_only.select_dtypes(include=[np.number]).mean()
        self.categorical_defaults = features_only.select_dtypes(exclude=[np.number]).mode().iloc[0]
        
        # Save exact encoded schema structures
        dummy_df = pd.get_dummies(features_only, columns=['Gender', 'College_Tier', 'Specialization'], dtype=int)
        self.expected_columns = dummy_df.columns.tolist()

    def preprocess_raw_input(self, raw_input_df):
        """Pipeline executing Missing Values, Deduplication, and Encoding sequentially."""
        df_step = raw_input_df.copy()
        
        # Module 1: Duplicate Removal
        df_step = df_step.drop_duplicates(keep='first')
        
        # Module 2: Missing Value Imputation
        for col in df_step.columns:
            if df_step[col].isnull().any():
                if col in self.numerical_defaults:
                    df_step[col] = df_step[col].fillna(self.numerical_defaults[col])
                elif col in self.categorical_defaults:
                    df_step[col] = df_step[col].fillna(self.categorical_defaults[col])
                    
        # Module 3: Categorical Encoding
        categorical_cols = df_step.select_dtypes(exclude=[np.number]).columns.tolist()
        if categorical_cols:
            df_step = pd.get_dummies(df_step, columns=categorical_cols, dtype=int)
            
        # Module 4: Schema Alignment (Ensures shapes match machine learning requirements)
        for col in self.expected_columns:
            if col not in df_step.columns:
                df_step[col] = 0
        df_step = df_step[self.expected_columns]
        
        return df_step

# Initialize our manager with our training domain knowledge
pipeline = PipelineManager(df_train)


# Step 4: Define Mock Prediction Model Interface
def mock_predict_model(processed_input_features):
    """Simulates a trained classification model mapping features to a binary prediction."""
    # Returns 1 (Placed) or 0 (Not Placed) for every record passed
    return np.array([1] * len(processed_input_features))


# Step 5: Simulate Live Raw User Inputs (With missing entries and raw strings)
raw_user_input = pd.DataFrame([{
    "Age": 20,
    "Gender": "Male",
    "College_Tier": "Tier 3",
    "Specialization": np.nan,  # Missing Categorical (should trigger mode fill)
    "CGPA": np.nan,            # Missing Numerical (should trigger mean fill)
    "DSA_Problems_Solved": 310,
    "Internships": 1,
    "Certifications": 4,
    "Projects_Count": 3,
    "Communication_Skills": 85,
    "Aptitude_Test_Score": 78,
    "LeetCode_Rating": 1590,
    "GitHub_Contributions": 120,
    "Hackathons_Participated": 2,
    "AI_ML_Skill_Level": 6,
    "System_Design_Knowledge": 5,
    "Resume_Score": 80,
    "Mock_Interview_Score": 75
}])

print("--- Incoming Raw User Input Record ---")
print(raw_user_input[['Gender', 'Specialization', 'CGPA']])


# Step 6: Automated Processing to Prediction Loop Execution
print("\n--- Running Preprocessing Pipeline on User Input ---")
clean_user_input = pipeline.preprocess_raw_input(raw_user_input)

print("\n--- Generating Model Prediction ---")
predictions = mock_predict_model(clean_user_input)
print(f"Prediction Output Array: {predictions}")


# Step 7: Automated Integration Verification Block
# Verify data shape matches structural expectations and contains 0 missing entries
is_aligned = clean_user_input.shape[1] == len(pipeline.expected_columns)
has_no_nulls = clean_user_input.isnull().sum().sum() == 0
has_predictions = len(predictions) == len(raw_user_input)

print("\n--- Test Verification Result ---")
print("Module Name: Pipeline Integration & Inference Module")
print("Objective: Verify integration of preprocessing with prediction.")
print("Expected Result: User input automatically preprocessed before prediction.")

if is_aligned and has_no_nulls and has_predictions:
    print("Actual Result: Preprocessing executed correctly.")
    print("Status: Pass ✅")
else:
    print("Actual Result: Integrity break between processing transformations and model ingestion.")
    print("Status: Fail ❌")


--- Incoming Raw User Input Record ---
  Gender  Specialization  CGPA
0   Male             NaN   NaN

--- Running Preprocessing Pipeline on User Input ---

--- Generating Model Prediction ---
Prediction Output Array: [1]

--- Test Verification Result ---
Module Name: Pipeline Integration & Inference Module
Objective: Verify integration of preprocessing with prediction.
Expected Result: User input automatically preprocessed before prediction.
Actual Result: Preprocessing executed correctly.
Status: Pass ✅
